# Building and Engineering Specialist RAG Agent <br>(Haystack + RAG + Memory)

**Build an AI Agent that helps developers:**
- fix bugs
- follow coding standards
- avoid repeated mistakes
- learn from past reviews & discussions

**Using Mem0 Memory Store with Haystack Agents:**<br>

[Mem0](https://mem0.ai/) is a managed memory layer for AI agents. Instead of passing entire conversation histories to an LLM on every turn, Mem0 intelligently extracts and compresses key facts from conversations into optimized memory representations.
Mem0 manages a cycle of **extraction**, **consolidation**, and **retrieval**. 

---


**Project Steps**

1. Set up a `Mem0MemoryStore` and add memories about a user
2. Inspect what Mem0 actually stored
3. Create a Haystack Agent that uses the memory store
4. Ask the Agent personalized questions and see how it leverages stored memories

## Install the required dependencies

In [1]:
import importlib.util, subprocess, sys

_pkgs = ["haystack-ai", "haystack-experimental", "mem0ai", "colorama"]
_missing = [p for p in _pkgs if importlib.util.find_spec(p.replace("-", "_").split("[")[0]) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _pkgs)

## Set up API keys

This notebook requires two API keys:
- **`OPENAI_API_KEY`**: Used by the [`OpenAIChatGenerator`](https://docs.haystack.deepset.ai/docs/openaichatgenerator) to power the Agent's LLM.
- **`MEM0_API_KEY`**: Used by `Mem0MemoryStore` to connect to the [Mem0 Platform](https://app.mem0.ai/). You can get a free API key by signing up.

In [2]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

# MEM0_USER_ID is optional — the notebook hardcodes "engineering_user" as the user scope
for key in ("OPENAI_API_KEY", "MEM0_API_KEY"):
    if not os.environ.get(key):
        raise EnvironmentError(f"{key} is not set. Add it to your .env file or environment.")

## RAG Setup + Memory 
[Creating Your First QA Pipeline with Retrieval-Augmentation](https://haystack.deepset.ai/tutorials/27_first_rag_pipeline)

### Setup Memory Layer (Mem0)
1. create a [new account](https://app.mem0.ai/login)
2. get an [API KEY](https://app.mem0.ai/dashboard/api-keys) and add to `env` file
3. [access dashboard](https://app.mem0.ai/dashboard) to view project logs and outputs

In [3]:
import os
from mem0 import MemoryClient

# add a memory store 
memory = MemoryClient(api_key=os.getenv("MEM0_API_KEY"))
MEMORY_USER_ID = "engineering_user"
MEMORY_TYPE = "engineering_feedback"

filters = {
   "OR":[
      {
        "user_id":MEMORY_USER_ID, 
      },
   ]
}



**Helper Functions**

In [4]:
def store_feedback(message, memory_type: str = MEMORY_TYPE):
    """Store feedback in memory with a specific type."""
    memory.add(
        message,
        user_id=MEMORY_USER_ID,           
        metadata={"type": memory_type}
    )

def retrieve_memory(user_input, limit=5):
    """Retrieve feedback from memory based on user input and filters."""
    memories = memory.search(
        user_input,
        filters=filters,       
        limit=limit
    )

    if isinstance(memories, dict):
        items = memories.get("results", [])
    elif isinstance(memories, list):
        items = memories
    else:
        items = []

    filtered_items = [
        m for m in items
        if isinstance(m, dict)
        and isinstance(m.get("metadata"), dict)
        and m.get("metadata", {}).get("type") == MEMORY_TYPE
    ]

    return "\n".join(
        m.get("memory", "")
        for m in filtered_items
        if m.get("memory")
    )

### Build RAG for Coding Standards

#### Load and Store Data (standards and best practices)

In [5]:
from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore

# Initialize in-memory store
doc_store = InMemoryDocumentStore()

def load_standards(file_path: str):
    """Load coding standards from a text file."""
    with open(file_path, "r", encoding="utf-8") as f:
        lines = [
            line.strip()
            for line in f.readlines()
            if line.strip() and not line.startswith("#")
        ]
    return lines

# Load standards from file
standards = load_standards("coding_standards.txt")

# Convert to Haystack Documents with metadata
docs = [
    Document(
        content=standard,
        meta={"type": "coding_standard"}
    )
    for standard in standards
]

# Write to document store
doc_store.write_documents(docs)

print(f"Loaded {len(docs)} coding standards.")

Loaded 15 coding standards.


### Create Embeddings and Retriever

In [6]:
from haystack.components.retrievers import InMemoryEmbeddingRetriever
from haystack.components.embedders import (
    OpenAIDocumentEmbedder,
    OpenAITextEmbedder,
)
from haystack.document_stores.types import DuplicatePolicy

embedder = OpenAIDocumentEmbedder(progress_bar=False)
embedded_docs = embedder.run(documents=docs)["documents"]
doc_store.write_documents(embedded_docs, policy=DuplicatePolicy.OVERWRITE)

query_embedder = OpenAITextEmbedder()
retriever = InMemoryEmbeddingRetriever(doc_store)

## Create Functions

In [7]:
# retrieve for past requests and feedback
def memory_function(query: str):
    return retrieve_memory(query)

# retrieve standards_tool("What are some best practices for software development?")
def standards_function(query: str):
    query_embedding = query_embedder.run(text=query)["embedding"]
    result = retriever.run(query_embedding=query_embedding, top_k=3)

    return "\n".join(doc.content for doc in result["documents"])


## Creating Tools

In [8]:
from haystack.tools import Tool

memory_parameters = {
    "type": "object",
    "properties": {
        "query": {
            "type": "string",
            "description": "Query to search past issues and engineering feedback"
        }
    },
    "required": ["query"],
    "additionalProperties": False
}

standards_parameters = {
    "type": "object",
    "properties": {
        "query": {
            "type": "string",
            "description": "Query to retrieve coding standards and best practices"
        }
    },
    "required": ["query"],
    "additionalProperties": False
}


memory_tool = Tool(
    name="engineering_memory",
    description="Retrieve past engineering issues and lessons learned.",
    parameters=memory_parameters,
    function=memory_function
)

standards_tool = Tool(
    name="coding_standards",
    description="Retrieve coding standards and best practices.",
    parameters=standards_parameters,
    function=standards_function
)

## Create Agent with LLM and instructions

In [9]:
from haystack.components.generators.chat.openai import OpenAIChatGenerator
from haystack.components.agents import Agent

llm = OpenAIChatGenerator(model="gpt-4o-mini")

engineering_agent = Agent(
    chat_generator=llm,
    tools=[memory_tool, standards_tool],
    system_prompt="""
You are a senior engineering specialist.

Use tools when useful.

Goals:
• recall past issues
• apply coding standards
• prevent failures
• propose scalable solutions

Always consider:
- performance
- scalability
- reliability
- production readiness
"""
)

In [10]:
# Quick schema sanity check (no API call)
assert memory_tool.parameters.get("type") == "object"
assert standards_tool.parameters.get("type") == "object"
print("Tool schemas are valid object JSON schemas.")

Tool schemas are valid object JSON schemas.


## Usage: Stream Responses in real-time

In [ ]:
from haystack.dataclasses import ChatMessage
from colorama import Fore, Style

def run(question: str):

    store_feedback(f"User asked: {question}")
    query_with_context = retrieve_memory(question) or question

    result = engineering_agent.run(
        messages=[ChatMessage.from_user(query_with_context)],
        memory_store_kwargs={"user_id": MEMORY_USER_ID},
        stream=True,
    )

    answer = result["last_message"].text
    store_feedback(f"Agent response: {answer}")
    print(answer)
    return answer

while True:
    user_input = input("Ask the engineering agent (or 'exit'/'quit' to stop): ")

    if user_input.lower() in {"exit", "quit"}:
        break
    if user_input:
        print(f"{Fore.CYAN}User input: {user_input}{Style.RESET_ALL}")
        print(f"\n{Fore.YELLOW}🔄 Processing ...{Style.RESET_ALL}")
        response = run(user_input)

Ask the engineering agent (or 'exit' to quit):  My async endpoint is slow under load, what should I check?


User input: My async endpoint is slow under load, what should I check?

🔄 Processing ...
To address the slow async endpoint under load, here's a summary of the recommendations and considerations based on past issues and coding standards:

### Key Areas to Check:

1. **Resource Monitoring:**
   - Monitor **CPU**, **memory**, and **network usage** during load testing. This helps detect any resource saturation that may contribute to performance issues.

2. **Performance Profiling:**
   - Conduct performance profiling to identify bottlenecks in the application. Focus on areas such as database queries, concurrency limits, and asynchronous operation implementation.

3. **Database Queries:**
   - Ensure that database queries are optimized and proper indexing is utilized to improve performance.

4. **Concurrency Limits:**
   - Review concurrency settings to ensure they are optimal for your load requirements. Use lightweight, non-blocking components to maintain responsiveness.

5. **Caching Str

Ask the engineering agent (or 'exit' to quit):  quit


User input: quit

🔄 Processing ...
To address the issues with your slow async endpoint under load, here's a comprehensive approach based on past engineering feedback and established coding standards:

### Key Recommendations
1. **Avoid Shared Mutable State**: This can lead to contention in concurrent systems, negatively impacting performance. Instead, use immutable data structures or manage state in a thread-safe way.

2. **Use Lightweight, Non-blocking Components**: Implement lightweight components that do not block operations, thus maintaining the responsiveness of your async service.

3. **Optimize Database Performance**:
   - Ensure SQL queries are optimized for performance.
   - Use proper indexing to improve query speeds.
   - Monitor and analyze query performance to identify bottlenecks.

4. **Load Testing**:
   - Conduct thorough load testing to gather performance data before making changes. This helps avoid premature optimization.
   - Monitor CPU, memory, and network usage du

## View Results in the Dashboard
[Dashboard](https://app.mem0.ai/dashboard)

In [ ]:
from IPython.display import Image, display

display(Image(filename="mem0-dashboard.png"))